In [ ]:
import os
import ast
import copy
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, top_k_accuracy_score

In [ ]:
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
drive_root = "/content/drive/MyDrive"

matches = []

for root, dirs, files in os.walk(drive_root):
    for f in files:
        if f in [
            "scenario23_img_beam.csv",
            "scenario23_pos_beam.csv",
            "scenario23.csv"
        ]:
            matches.append(os.path.join(root, f))

print("Found full CSV files:")
for m in matches:
    print(m)

In [ ]:
FULL_CSV_PATH = "PASTE_FULL_CSV_PATH_HERE"

full_df = pd.read_csv(FULL_CSV_PATH)

print("Shape:", full_df.shape)
display(full_df.head())
print("Columns:", full_df.columns.tolist())

In [ ]:
required_cols = ["index", "unit2_pos", "unit1_beam"]

for col in required_cols:
    assert col in full_df.columns, f"Missing column: {col}"

full_df = full_df.sort_values("index").reset_index(drop=True)

print(full_df[required_cols].head())
print(full_df[required_cols].tail())

print("Total samples:", len(full_df))
print("Index min:", full_df["index"].min())
print("Index max:", full_df["index"].max())

In [ ]:
idx = full_df["index"].astype(int).values
diffs = np.diff(idx)

unique_diffs, counts = np.unique(diffs, return_counts=True)

print("Unique index differences:", unique_diffs, counts)
print("Continuous +1 transitions:", np.sum(diffs == 1))
print("Broken transitions:", np.sum(diffs != 1))

position feature function

In [ ]:
def parse_unit2_pos(pos_value):
    if isinstance(pos_value, str):
        return ast.literal_eval(pos_value)
    return pos_value


def add_position_features(df):
    df = df.copy()

    parsed = df["unit2_pos"].apply(parse_unit2_pos)

    df["pos_x"] = parsed.apply(lambda v: float(v[0]))
    df["pos_y"] = parsed.apply(lambda v: float(v[1]))

    eps = 1e-8

    df["distance"] = np.sqrt(df["pos_x"]**2 + df["pos_y"]**2)
    df["distance2"] = df["distance"] ** 2
    df["distance3"] = df["distance"] ** 3

    df["angle"] = np.arctan2(df["pos_y"], df["pos_x"])
    df["sin_angle"] = np.sin(df["angle"])
    df["cos_angle"] = np.cos(df["angle"])

    df["pos_x2"] = df["pos_x"] ** 2
    df["pos_y2"] = df["pos_y"] ** 2
    df["pos_x3"] = df["pos_x"] ** 3
    df["pos_y3"] = df["pos_y"] ** 3

    df["pos_xy"] = df["pos_x"] * df["pos_y"]

    df["unit_x"] = df["pos_x"] / (df["distance"] + eps)
    df["unit_y"] = df["pos_y"] / (df["distance"] + eps)

    df["sin2_angle"] = np.sin(2 * df["angle"])
    df["cos2_angle"] = np.cos(2 * df["angle"])
    df["sin3_angle"] = np.sin(3 * df["angle"])
    df["cos3_angle"] = np.cos(3 * df["angle"])

    df["dist_sin"] = df["distance"] * df["sin_angle"]
    df["dist_cos"] = df["distance"] * df["cos_angle"]

    return df

In [ ]:
full_df_fe = add_position_features(full_df)

feature_cols = [
    "pos_x", "pos_y",
    "distance", "distance2", "distance3",
    "angle",
    "sin_angle", "cos_angle",
    "pos_x2", "pos_y2",
    "pos_x3", "pos_y3",
    "pos_xy",
    "unit_x", "unit_y",
    "sin2_angle", "cos2_angle",
    "sin3_angle", "cos3_angle",
    "dist_sin", "dist_cos"
]

label_col = "unit1_beam"

print("Feature count:", len(feature_cols))
print("Label unique:", full_df_fe[label_col].nunique())
print("Label min/max:", full_df_fe[label_col].min(), full_df_fe[label_col].max())

display(full_df_fe[["index", "unit2_pos", *feature_cols, label_col]].head())

In [ ]:
all_labels = sorted(full_df_fe[label_col].astype(int).unique())

label_to_id = {label: i for i, label in enumerate(all_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

num_classes = len(all_labels)

print("num_classes:", num_classes)
print("label_to_id:", label_to_id)

Paper style sequence build 